In [2]:
# ── Fine-tuning LaBSE for aspect classification — Colab GPU ──
# All imports and setup in one cell.

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

# ── Core ──
import os
import numpy as np
import pandas as pd

# ── Deep Learning ──
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ── Transformers ──
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

# ── Metrics ──
from sklearn.metrics import f1_score, precision_score, recall_score

# ── Reproducibility (same seed as the frozen model) ──
SEED = 42
import random
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ASPECTS = ["service", "ambience", "ingredients", "experience"]

print("Setup done. Device:", DEVICE)


CUDA available: True
GPU: Tesla T4
Setup done. Device: cuda


In [3]:
# ── Load the exact split exported from the project notebook ──
# CSVs uploaded to Colab root; locate them wherever they landed.
for base in ["/content", "/", os.getcwd()]:
    if os.path.exists(os.path.join(base, "finetune_train.csv")):
        break
print("CSVs found in:", base)

train = pd.read_csv(os.path.join(base, "finetune_train.csv"))
val   = pd.read_csv(os.path.join(base, "finetune_val.csv"))
test  = pd.read_csv(os.path.join(base, "finetune_test.csv"))

for name, df in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:6s} {len(df):5d} rows   aspect labels: {int(df[ASPECTS].values.sum())}")
print("\nColumns:", list(train.columns))
print("Per-aspect (train):", {a: int(train[a].sum()) for a in ASPECTS})

CSVs found in: /
train   1192 rows   aspect labels: 1459
val      255 rows   aspect labels: 307
test     256 rows   aspect labels: 311

Columns: ['text', 'lang', 'service', 'ambience', 'ingredients', 'experience', 'sentiment']
Per-aspect (train): {'service': 801, 'ambience': 330, 'ingredients': 183, 'experience': 145}


In [4]:
# ── The fine-tuning model: LaBSE (unfrozen) + aspect head ──
# Unlike the frozen project model, the encoder's weights train too.

MODEL_NAME = "sentence-transformers/LaBSE"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class LaBSEAspectClassifier(nn.Module):
    def __init__(self, n_aspects=4, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)   # NOT frozen
        hidden = self.encoder.config.hidden_size               # 768
        self.head = nn.Sequential(
            nn.Linear(hidden, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, n_aspects),
        )

    def mean_pool(self, out, mask):
        # LaBSE uses mean pooling over tokens (same as your frozen embed step)
        emb = out.last_hidden_state
        mask = mask.unsqueeze(-1).float()
        return (emb * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.mean_pool(out, attention_mask)
        return self.head(pooled)

model = LaBSEAspectClassifier(n_aspects=len(ASPECTS)).to(DEVICE)

n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model built. Total params: {n_total:,}")
print(f"Trainable params: {n_train:,}  (encoder IS training — this is the fine-tune)")

config.json:   0%|          | 0.00/804 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/5.22M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.62M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.88GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model built. Total params: 471,124,740
Trainable params: 471,124,740  (encoder IS training — this is the fine-tune)


In [5]:
# ── Tokenize and batch. Aspect targets are the 4 binary columns. ──
MAX_LEN = 128   # your §3 scratch check showed almost no reviews exceed this

class AspectDataset(Dataset):
    def __init__(self, df):
        self.texts = df["text"].tolist()
        self.labels = df[ASPECTS].values.astype("float32")
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        enc = tokenizer(self.texts[i], truncation=True, max_length=MAX_LEN,
                        padding="max_length", return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[i]),
        }

g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(AspectDataset(train), batch_size=16, shuffle=True, generator=g)
val_loader   = DataLoader(AspectDataset(val),   batch_size=32)
test_loader  = DataLoader(AspectDataset(test),  batch_size=32)

# Class weights for the rarer aspects (same idea as the frozen head's weighting)
pos = train[ASPECTS].sum().values
neg = len(train) - pos
pos_weight = torch.tensor(neg / pos, dtype=torch.float32).to(DEVICE)
print("Batches:", len(train_loader), "train /", len(val_loader), "val /", len(test_loader), "test")
print("pos_weight per aspect:", {a: round(float(w),2) for a,w in zip(ASPECTS, pos_weight)})

Batches: 75 train / 8 val / 8 test
pos_weight per aspect: {'service': 0.49, 'ambience': 2.61, 'ingredients': 5.51, 'experience': 7.22}


In [6]:
# ── Fine-tune LaBSE end-to-end. Checkpoint best val macro-F1. ──
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

EPOCHS = 4   # fine-tuning a transformer needs few epochs; more risks overfitting 1,192 rows
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1*total_steps), total_steps)

@torch.no_grad()
def evaluate(loader):
    model.eval()
    P, T = [], []
    for b in loader:
        logits = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
        P.append((torch.sigmoid(logits) >= 0.5).int().cpu())
        T.append(b["labels"].int())
    P, T = torch.cat(P).numpy(), torch.cat(T).numpy()
    return f1_score(T, P, average="macro", zero_division=0), P, T

best_f1, best_state = -1, None
for epoch in range(1, EPOCHS+1):
    model.train()
    losses = []
    for b in train_loader:
        optimizer.zero_grad()
        logits = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
        loss = loss_fn(logits, b["labels"].to(DEVICE))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        losses.append(loss.item())
    val_f1, _, _ = evaluate(val_loader)
    print(f"epoch {epoch}  train_loss {np.mean(losses):.4f}  val macro-F1 {val_f1:.3f}")
    if val_f1 > best_f1:
        best_f1, best_state = val_f1, {k: v.cpu().clone() for k,v in model.state_dict().items()}

print(f"\nBest val macro-F1: {best_f1:.3f}  (frozen head was 0.588 on val)")
model.load_state_dict(best_state)

epoch 1  train_loss 0.9153  val macro-F1 0.543
epoch 2  train_loss 0.6913  val macro-F1 0.625
epoch 3  train_loss 0.4694  val macro-F1 0.775
epoch 4  train_loss 0.3447  val macro-F1 0.804

Best val macro-F1: 0.804  (frozen head was 0.588 on val)


<All keys matched successfully>

In [7]:
# ── Fine-tuned model on the HELD-OUT TEST set (the honest number) ──
test_f1, P_test, T_test = evaluate(test_loader)
print(f"Fine-tuned test macro-F1: {test_f1:.3f}   (frozen head was 0.611 on test)\n")

# Per-aspect on test
print(f"{'aspect':14s} {'prec':>6s} {'recall':>7s} {'F1':>6s}   (frozen F1)")
frozen_f1 = {"service":0.833, "ambience":0.602, "ingredients":0.552, "experience":0.456}
for j, a in enumerate(ASPECTS):
    p = precision_score(T_test[:,j], P_test[:,j], zero_division=0)
    r = recall_score(T_test[:,j], P_test[:,j], zero_division=0)
    f = f1_score(T_test[:,j], P_test[:,j], zero_division=0)
    print(f"{a:14s} {p:6.3f} {r:7.3f} {f:6.3f}   ({frozen_f1[a]:.3f})")

# Per-language macro-F1 on test
print("\nPer-language macro-F1 (test):")
frozen_lang = {"en":0.564, "es":0.679, "pt":0.518, "it":0.671}
langs = test["lang"].values
for lang in ["en","es","pt","it"]:
    m = langs == lang
    if m.sum():
        f = f1_score(T_test[m], P_test[m], average="macro", zero_division=0)
        print(f"  {lang}  {f:.3f}   (frozen {frozen_lang[lang]:.3f})")

Fine-tuned test macro-F1: 0.808   (frozen head was 0.611 on test)

aspect           prec  recall     F1   (frozen F1)
service         0.939   0.863  0.900   (0.833)
ambience        0.688   0.821  0.748   (0.602)
ingredients     0.824   0.913  0.866   (0.552)
experience      0.683   0.757  0.718   (0.456)

Per-language macro-F1 (test):
  en  0.702   (frozen 0.564)
  es  0.862   (frozen 0.679)
  pt  0.783   (frozen 0.518)
  it  0.839   (frozen 0.671)
